# 04 - Modèle avec contexte

Ce notebook entraîne un modèle NCF qui intègre des features contextuelles extraites de MovieLens 100K.

L'objectif est de comparer une architecture baseline sans contexte avec un modèle qui utilise l'heure, le jour de la semaine, et des features de session.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader

sys.path.append(str((Path('..') / 'src').resolve()))
from data_utils import load_movielens_100k, encode_ids
from context_features import extract_all_context_features
from models import NCFContext
from metrics import hit_rate, ndcg, mrr

print('Python:', sys.version.split()[0])
print('pandas :', pd.__version__)
print('numpy :', np.__version__)
print('torch :', torch.__version__)


## Chargement et extraction des features de contexte

Nous utilisons MovieLens 100K en chargeant le fichier brut et en extrayant les features temporelles et de session.


In [ ]:
NOTEBOOK_ROOT = Path('.')
RAW_ROOT = (NOTEBOOK_ROOT / '..' / 'data' / 'raw' / 'movielens' / 'ml-100k').resolve()
print('RAW_ROOT =', RAW_ROOT)

df = load_movielens_100k(RAW_ROOT)
print('raw shape:', df.shape)

df = encode_ids(df)
df = extract_all_context_features(df)

print('processed shape:', df.shape)
print(df.columns.tolist())
print(df[['user_id_encoded', 'item_id_encoded', 'hour_of_day', 'dow_sin', 'session_length', 'session_position_norm']].head())


### Features contextuelles utilisées

Nous utilisons un ensemble de features continues extraites pour représenter l'heure, le jour de la semaine, et le comportement de session.


In [ ]:
context_cols = [
    'hour_sin', 'hour_cos',
    'dow_sin', 'dow_cos',
    'is_weekend',
    'time_since_last_interaction_log',
    'session_length',
    'session_position_norm'
]
print('Context columns:', context_cols)
print(df[context_cols].head())


## Préparation des labels

Nous convertissons les notes en labels binaires pour entraîner un modèle de ranking simple.


In [ ]:
df['label'] = (df['rating'] >= 4).astype(int)
print(df['label'].value_counts(normalize=True))

n_users = df['user_id_encoded'].nunique()
n_items = df['item_id_encoded'].nunique()
context_dim = len(context_cols)

print('Users:', n_users, 'Items:', n_items, 'Context dim:', context_dim)


## Split train/test

Nous utilisons un split aléatoire en 80/20. On pourra remplacer par un split temporel spécifique plus tard.


In [ ]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
print('train shape:', train_df.shape)
print('test shape:', test_df.shape)


## Dataset PyTorch avec contexte

Le dataset renvoie l'item, l'utilisateur, le contexte et le label.


In [ ]:
class ContextMovielensDataset(Dataset):
    def __init__(self, df, context_columns):
        self.user_ids = torch.tensor(df['user_id_encoded'].values, dtype=torch.long)
        self.item_ids = torch.tensor(df['item_id_encoded'].values, dtype=torch.long)
        self.contexts = torch.tensor(df[context_columns].values, dtype=torch.float32)
        self.labels = torch.tensor(df['label'].values, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.user_ids[idx], self.item_ids[idx], self.contexts[idx], self.labels[idx]

train_dataset = ContextMovielensDataset(train_df, context_cols)
test_dataset = ContextMovielensDataset(test_df, context_cols)

batch_size = 256
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print('Train batches:', len(train_loader))
print('Test batches:', len(test_loader))


## Construction du modèle contextuel

Nous utilisons `NCFContext` avec une fusion par concaténation. Cet exemple peut être étendu ultérieurement avec `film` ou `attention`.


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

model = NCFContext(
    num_users=n_users,
    num_items=n_items,
    context_dim=context_dim,
    embed_dim=32,
    mlp_layers=[64, 32, 16],
    context_embed_dim=32,
    dropout=0.2,
    fusion_type='concat'
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.BCEWithLogitsLoss()

print(model)


## Entraînement

Nous entraînons le modèle avec une boucle simple et surveillons la perte d'entraînement.


In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for user_batch, item_batch, context_batch, labels in loader:
        user_batch = user_batch.to(device)
        item_batch = item_batch.to(device)
        context_batch = context_batch.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(user_batch, item_batch, context_batch)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)

    return total_loss / len(loader.dataset)

epochs = 8
history = []
for epoch in range(1, epochs + 1):
    loss = train_epoch(model, train_loader, optimizer, criterion, device)
    history.append(loss)
    print(f'Epoch {epoch}/{epochs} - train loss: {loss:.4f}')


## Évaluation ranking

Nous évaluons sur un sous-ensemble de test pour maîtriser le temps d'inférence.


In [ ]:
eval_df = test_df.sample(n=min(1000, len(test_df)), random_state=42).reset_index(drop=True)
all_item_ids = torch.arange(n_items, dtype=torch.long, device=device)

def evaluate_ranking(model, df, k=10):
    model.eval()
    hits, ndcgs, mrrs = [], [], []
    with torch.no_grad():
        for _, row in df.iterrows():
            user_id = torch.tensor([row['user_id_encoded']], dtype=torch.long, device=device)
            item_id = int(row['item_id_encoded'])
            context = torch.tensor(row[context_cols].values, dtype=torch.float32, device=device).unsqueeze(0)
            user_batch = user_id.expand(n_items)
            context_batch = context.expand(n_items, context_dim)
            logits = model(user_batch, all_item_ids, context_batch).cpu().numpy()
            ranking = np.argsort(-logits)
            hits.append(int(item_id in ranking[:k]))
            ndcgs.append(ndcg(ranking, item_id, k))
            mrrs.append(mrr(ranking, item_id))
    return np.mean(hits), np.mean(ndcgs), np.mean(mrrs)

hit10, ndcg10, mrr_score = evaluate_ranking(model, eval_df, k=10)
print(f'Hit Rate@10: {hit10:.4f}')
print(f'NDCG@10: {ndcg10:.4f}')
print(f'MRR: {mrr_score:.4f}')


## Sauvegarde du modèle contextuel

Nous sauvegardons les poids pour une comparaison avec le baseline.


In [ ]:
RESULTS_DIR = (NOTEBOOK_ROOT / '..' / 'results' / 'models').resolve()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = RESULTS_DIR / 'context_ncf_ml100k.pt'
torch.save(model.state_dict(), MODEL_PATH)
print('Saved context model to', MODEL_PATH)


## Résumé

Ce notebook entraîne un NCF enrichi par des features contextuelles sur MovieLens 100K.
La comparaison avec le baseline permettra d'évaluer la valeur ajoutée des données temporelles et de session.
